In [1]:
# ================= HEART FAILURE — BEST MODEL + mAP ===================
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import accuracy_score,f1_score,roc_auc_score, average_precision_score  # ← mAP imported

from imblearn.over_sampling import SMOTE,ADASYN
from imblearn.combine import SMOTETomek

np.random.seed(42)
df = pd.read_csv("heartfailure.csv")

# ---------------- Target & Features ----------------
y = df["HeartDisease"].astype(int)
X = pd.get_dummies(df.drop(columns=["HeartDisease"]), drop_first=True)
X = SimpleImputer(strategy="median").fit_transform(X)

# ---------------- Balancing Techniques ----------------
methods = {
    "SMOTE":SMOTE(random_state=42),
    "ADASYN":ADASYN(random_state=42),
    "TOMEK":SMOTETomek(random_state=42)
}

models = {
    "XGB": XGBClassifier(eval_metric="logloss"),
    "LightGBM": LGBMClassifier(class_weight="balanced"),
    "CatBoost": CatBoostClassifier(verbose=0),
    "BalancedRF": BalancedRandomForestClassifier(n_estimators=600),
    "GradientBoost": GradientBoostingClassifier(),
    "RandomForest": RandomForestClassifier(class_weight="balanced")
}

best_model = None
best_auc = -1
best_name = ""
best_metrics = {}                  # ← mAP will be stored here

print("\n=== HEART FAILURE RESULTS — SELECTING BEST MODEL ===\n")

for m,resamp in methods.items():
    Xr, yr = resamp.fit_resample(X, y)
    Xtr,Xte, ytr,yte = train_test_split(Xr, yr, test_size=0.2, stratify=yr, random_state=42)

    # Feature Selection
    fs = SelectKBest(mutual_info_classif, k=min(20, Xtr.shape[1]))
    Xtr2, Xte2 = fs.fit_transform(Xtr, ytr), fs.transform(Xte)

    print(f"\n--- [{m}] ---")
    for name,model in models.items():
        model.fit(Xtr2, ytr)

        prob = model.predict_proba(Xte2)[:,1]
        pred = (prob>0.5).astype(int)

        acc = accuracy_score(yte,pred)
        f1  = f1_score(yte,pred)
        auc = roc_auc_score(yte,prob)
        mAP = average_precision_score(yte,prob)        # ← added

        print(f"{name:<15} | ACC={acc:.4f} | F1={f1:.4f} | AUC={auc:.4f} | mAP={mAP:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_name = f"{name} + {m}"
            best_model = model
            best_metrics = {"ACC":acc,"F1":f1,"AUC":auc,"mAP":mAP}  # ← store best mAP here

# ---------------- FINAL OUTPUT ONLY ----------------
print(f"\n\n🔥 BEST HEART FAILURE MODEL ⇒ {best_name}")
print(f"📌 AUC = {best_metrics['AUC']:.4f}")
print(f"💥 mAP = {best_metrics['mAP']:.4f}")    # ← REQUIRED OUTPUT



=== HEART FAILURE RESULTS — SELECTING BEST MODEL ===


--- [SMOTE] ---
XGB             | ACC=0.8922 | F1=0.8900 | AUC=0.9465 | mAP=0.9400
[LightGBM] [Info] Number of positive: 406, number of negative: 406
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000544 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 573
[LightGBM] [Info] Number of data points in the train set: 812, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
LightGBM        | ACC=0.8922 | F1=0.8900 | AUC=0.9519 | mAP=0.942

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


CatBoost        | ACC=0.9167 | F1=0.9163 | AUC=0.9588 | mAP=0.9445
BalancedRF      | ACC=0.9020 | F1=0.9020 | AUC=0.9514 | mAP=0.9406
GradientBoost   | ACC=0.8922 | F1=0.8889 | AUC=0.9516 | mAP=0.9401
RandomForest    | ACC=0.9069 | F1=0.9064 | AUC=0.9500 | mAP=0.9372

--- [ADASYN] ---
XGB             | ACC=0.8737 | F1=0.8812 | AUC=0.9300 | mAP=0.9383
[LightGBM] [Info] Number of positive: 406, number of negative: 351
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000190 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 410
[LightGBM] [Info] Number of data points in the train set: 757, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM]

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


CatBoost        | ACC=0.9158 | F1=0.9208 | AUC=0.9442 | mAP=0.9501
BalancedRF      | ACC=0.8789 | F1=0.8867 | AUC=0.9358 | mAP=0.9404
GradientBoost   | ACC=0.8947 | F1=0.9029 | AUC=0.9318 | mAP=0.9435
RandomForest    | ACC=0.8579 | F1=0.8683 | AUC=0.9329 | mAP=0.9315

--- [TOMEK] ---
XGB             | ACC=0.9061 | F1=0.9091 | AUC=0.9540 | mAP=0.9485
[LightGBM] [Info] Number of positive: 362, number of negative: 361
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000186 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 563
[LightGBM] [Info] Number of data points in the train set: 723, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] 

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


CatBoost        | ACC=0.9171 | F1=0.9189 | AUC=0.9608 | mAP=0.9549
BalancedRF      | ACC=0.9171 | F1=0.9198 | AUC=0.9532 | mAP=0.9451
GradientBoost   | ACC=0.9006 | F1=0.9011 | AUC=0.9512 | mAP=0.9518
RandomForest    | ACC=0.9116 | F1=0.9140 | AUC=0.9526 | mAP=0.9355


🔥 BEST HEART FAILURE MODEL ⇒ CatBoost + TOMEK
📌 AUC = 0.9608
💥 mAP = 0.9549
